# MSCAD multi-stage intrusion-detection pipeline

Step 1 (PCAPNG -> one flow CSV per capture) lives in
`convert_pcapng_to_csvflows.ipynb`. This notebook starts at **Stage 2**.

| Stage | Module | What it does |
|-------|--------|--------------|
| **2** | `src\mscad\data_loading.py` | **load & merge the flow CSVs** (this notebook, below) |
| 3 | `src\mscad\preprocessing.py` | clean, encode, scale |
| 4 | `src\mscad\feature_selection.py` | Mutual Information / Permutation Importance |
| 5 | `src\mscad\autoencoder.py` | autoencoder feature extraction |
| 6 | `src\mscad\imbalance.py` | class-imbalance handling |
| 7 | `src\mscad\models.py` | RF, XGBoost, LSTM, CNN, hybrid CNN-LSTM |
| 8 | `src\mscad\evaluation.py`, `windows.py` | metrics; sliding windows, PrefixSpan, attack-state graph |

Every cell is safe to re-run, and each one calls a single function from
`data_loading`. All the logic is in that module, not here.

## 1. Setup

In [ ]:
import sys
from pathlib import Path

# The notebooks live in PROJECT_ROOT\notebooks, so when Jupyter is started in
# that folder the project root is one level up.  Starting Jupyter in the project
# root instead also works.  Nothing is hard-coded, so this is correct whether the
# code sits in C:\MSCAD2022\Notebook or in a git clone.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

%load_ext autoreload
%autoreload 2

import config

print("project root :", PROJECT_ROOT)
print("kernel python:", sys.version.split()[0])
print("flow CSVs in :", config.CSV_OUTPUT_DIR)
print("merged out   :", config.MERGED_DIR)
print("step 2 out   :", config.STEP2_OUTPUT_DIR)
print("kaggle ref   :", config.KAGGLE_CSV_DIR,
      "(exists)" if config.KAGGLE_CSV_DIR.is_dir() else "(not present - section 13 is skipped)")
print("random seed  :", config.RANDOM_SEED)

---
# Stage 2: Load & merge flow CSVs

**What this stage does**

Reads the six per-capture CSVs Step 1 wrote (`<stem>_flows.csv`, 73 columns:
6 flow identifiers + 66 MSCAD features + `Label`), checks each header, and
concatenates them into one dataset with three bookkeeping columns added:

| Added column | Meaning |
|---|---|
| `Source_Capture` | which capture the row came from, taken from the file name |
| `Orig_Row` | the row's position inside its own file, so any row can be traced back |
| `Timestamp_dt` | the `Timestamp` text parsed to a real datetime; the text column is kept |

76 columns in total. Rows are then put in **time order** with a stable sort, so
the merged file reads like the attack session it came from — which is what the
Stage 8 sequence mining and attack-state graph need.

**What this stage deliberately does NOT do**

No cleaning: `inf`, `-inf`, `NaN` and duplicate rows are all carried through
exactly as they are, and the class imbalance is left alone. No encoding, no
scaling, no train/test split, no feature selection, no windowing, no training.
Those are Stage 3 onward. Step 2's only job is to produce one faithful merged
dataset and describe it honestly.

In [ ]:
from src.mscad import data_loading

print(data_loading.__file__)

## 2.1 Find the flow CSVs

Only files matching `config.FLOW_CSV_PATTERN` (`*_flows.csv`) are picked up, so
`CSVs\raw\<stem>_raw.csv` is ignored — `glob()` does not recurse, so the `raw\`
folder is never even looked at.

`config.CSV_OUTPUT_DIR` is tried first; if it holds no matching files the other
`config.CSV_CANDIDATE_DIRS` are tried and **the folder actually used is printed**.
If the six captures of `config.CAPTURE_ORDER` are not all there, this raises
rather than merging an incomplete dataset.

In [ ]:
files, csv_folder = data_loading.find_flow_csvs()

## 2.2 Load each file and validate its header

Identifiers and `Label` as text, ports and `Protocol` as `int64`, the 66 features
as `float64` (which is what keeps `inf` as `inf`). The `Timestamp` text column is
kept untouched and `Timestamp_dt` is added beside it.

The header must equal `config.FLOW_COLUMNS` — same names, same order. A mismatch
raises with the exact diff, because quietly reordering columns here would corrupt
every later stage.

If the timestamp layout were ambiguous (e.g. `01/02/2017`, where day and month
cannot be told apart), loading stops and asks. Step 1 writes year-first
timestamps, so it should report ISO 8601 and carry on.

In [ ]:
frames, infos = {}, []
for f in files:
    df_one, info = data_loading.load_one_csv(f.path, capture=f.capture)
    data_loading.validate_columns(df_one, source=f.path.name)
    frames[f.capture] = df_one
    infos.append(info)

print()
print("timestamp format :", infos[0]["timestamp_format"])
print("why              :", infos[0]["timestamp_reason"])

## 2.3 Merge

Adds `Source_Capture` and `Orig_Row`, concatenates in `config.CAPTURE_ORDER`
with `ignore_index=True`, then sorts by `Timestamp_dt` with
`kind="mergesort"` — a **stable** sort, so rows sharing a timestamp keep their
original relative order and the result is reproducible rather than dependent on
pandas internals. Finally the columns are put in `config.MERGED_COLUMNS` order.

In [ ]:
merged, _ = data_loading.merge_flow_csvs(frames=frames)

## 2.4 Merge checks — PASS/FAIL with numbers

Nothing is called correct without a number beside it. Checked: total rows vs the
sum of the file rows; the row count kept for each capture; every `Label` count;
the column count, names and order; and the total `NaN` and `inf` cells across
the 66 features.

`all_passed` must be `True` before the merged file is used for anything.

In [ ]:
checks, all_passed = data_loading.check_merge(merged, infos)

print()
print("ALL MERGE CHECKS PASSED:", all_passed)
assert all_passed, "the merge is not faithful -- see the FAIL rows above"

## 2.5 Save

Writes `data\merged\MSCAD_merged.csv` and, when `pyarrow` is installed,
`MSCAD_merged.parquet` as well (if it is missing you get the `pip` command and
the CSV is written anyway). An existing file is kept unless
`config.OVERWRITE_MERGED = True`. The per-file numbers and every PASS/FAIL row
also go to `outputs\step2\merge_report.csv`.

In [ ]:
saved = data_loading.save_merged(merged, checks=checks, infos=infos)
saved

## 2.6 Load the merged dataset back from disk

Deliberately read back **from the file**, not reused from memory: that is what
proves the saved dataset really round-trips, `inf` and `NaN` included.

In [ ]:
dataset = data_loading.load_merged()

print("same shape as the in-memory merge:", dataset.shape == merged.shape)
dataset.shape

## 2.7 Overview of the merged dataset

Every section below renders as a **table**, not as printed text:

| # | Section |
|---|---------|
| 1 | shape and memory, with the 2 GB verdict |
| 2-3 | `head(10)` and `tail(5)`, all 76 columns, scroll sideways |
| 4 | one row per column: dtype, non-null, null |
| 5 | Label counts, percentages and the imbalance ratio |
| 6 | `Source_Capture` x `Label` |
| 7 | first/last timestamp and duration per capture |
| 8 | unique IPs and ports, then the protocol table (6 = TCP, 17 = UDP, 1 = ICMP) |
| 9 | NaN and inf per column, listing only the non-zero ones |
| 10 | duplicate counts |
| 11 | `describe()` for eight key features, plus an `inf_count` column |
| 12 | the Label bar chart, log scale |
| 13 | our Label counts beside the Kaggle file's, for reference only |

Section 11 computes its statistics on a **copy** with `inf` replaced by `NaN`, so
the numbers are readable and numpy stops emitting "invalid value encountered in
subtract" warnings. The `inf_count` column keeps the real `inf` counts visible,
and the dataset itself is unchanged.

Section 13 reads `config.KAGGLE_CSV_DIR`. If that folder or file is missing it
prints one line and carries on - Step 2 does not depend on it. Labels are
matched on a normalised form (trimmed, lower-cased, spaces and underscores
equivalent) but **nothing is renamed**: both spellings are shown, and labels
present on only one side get their own table.

Everything here **counts and describes** - nothing is dropped, filled or changed.

In [ ]:
summary = data_loading.summarize_dataset(dataset)

## 2.8 What Stage 3 has to deal with

Step 2 leaves the data exactly as Step 1 produced it, so these are now visible
and all still open:

1. **`inf` in the rate columns.** `Flow Byts/s` and `Flow Pkts/s` are `inf` for
   zero-duration (single-packet) flows — the mathematically correct answer, and
   the reason `describe()` shows `inf` for their mean and std. Scikit-learn and
   Keras both reject `inf`, so Stage 3 must decide: clip to a large finite value,
   replace with `NaN` then impute, or drop the rows. Counted in section 9.
2. **`NaN`.** Wherever section 9 shows non-zero counts, Stage 3 needs an explicit
   imputation choice per column — not a blanket `fillna(0)`, which would invent a
   zero-rate flow.
3. **Duplicate and identical-feature rows.** Section 10 counts them. In floods,
   port scans and brute force, thousands of flows genuinely look identical, so
   these are real observations, not data errors. Dropping them would delete the
   attack signal; keeping them all risks leaking near-identical rows across a
   train/test split. Stage 3 has to choose, with the split in mind.
4. **Class imbalance.** Section 5 prints the largest:smallest class ratio. That
   is what Stage 6 (`imbalance.py`) addresses — and it is also why accuracy will
   be a useless metric and macro-F1 / per-class recall are the ones to watch.
5. **The identifier columns.** `Src IP`, `Dst IP`, `Timestamp` and the ports are
   kept because Stage 8 needs them for sequences and the attack-state graph, but
   they must **not** be fed to a classifier as features: the attacker IPs decided
   the labels in Step 1, so training on them would be label leakage and would
   score near-perfectly while learning nothing.

---
# Stage 3 - preprocessing: clean, encode, scale

_(no code yet)_

---
# Stage 4 - feature selection (Mutual Information / Permutation Importance)

_(no code yet)_

---
# Stage 5 - autoencoder feature extraction

_(no code yet)_

---
# Stage 6 - class-imbalance handling

_(no code yet)_

---
# Stage 7 - models: RF, XGBoost, LSTM, CNN, hybrid CNN-LSTM

_(no code yet)_

---
# Stage 8 - evaluation, sliding windows, PrefixSpan, attack-state graph

This is why Step 1 and Step 2 keep Src IP / Dst IP / ports / protocol / Timestamp.

_(no code yet)_